# Endpoint Deployment: Idempotent Bundle Deploy

This notebook deploys a serving endpoint using Databricks Asset Bundle (DAB).

## Idempotency
- ✅ `databricks bundle deploy` is **idempotent by design**
- ✅ Creates endpoint if it doesn't exist
- ✅ Updates endpoint config if it exists
- ✅ No-op if config matches existing state
- ✅ Safe to re-run multiple times

## What Bundle Deploy Does
1. Validates bundle configuration
2. Creates/updates serving endpoint resource
3. Uploads model artifacts if needed
4. Configures endpoint settings (size, scale-to-zero, env vars)
5. Waits for endpoint to be ready (if new)

In [0]:
# Databricks notebook source
import subprocess
import sys
import os
from datetime import datetime

# Get parameters from job or widgets
dbutils.widgets.text("target", "qa", "Target Environment (qa or prod)")
target = dbutils.widgets.get("target")

print(f"{'='*70}")
print(f"  Deploying Serving Endpoint to {target.upper()}")
print(f"{'='*70}")
print(f"⏰ Started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

In [0]:
def run_shell_command(cmd, cwd=None):
    """
    Run a shell command and return output
    Raises exception if command fails
    """
    print(f"\n🔨 Running: {cmd}")
    print(f"   Working directory: {cwd or os.getcwd()}")
    
    try:
        result = subprocess.run(
            cmd,
            shell=True,
            cwd=cwd,
            capture_output=True,
            text=True,
            check=True
        )
        
        if result.stdout:
            print(result.stdout)
        
        return result.returncode
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Command failed with exit code {e.returncode}")
        if e.stdout:
            print(f"\nStdout:\n{e.stdout}")
        if e.stderr:
            print(f"\nStderr:\n{e.stderr}")
        raise

print("✅ Helper functions loaded")

In [0]:
# Get deployment directory
# This notebook is in /deployment/notebooks/, so parent is /deployment/
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
print(f"📝 Notebook path: {notebook_path}")

# Deployment directory is parent of notebooks directory
deployment_dir = "/Workspace" + notebook_path.rsplit("/", 2)[0]
print(f"📁 Deployment directory: {deployment_dir}")

# Verify databricks.yml exists
import os
if not os.path.exists(f"{deployment_dir}/databricks.yml"):
    print(f"❌ ERROR: databricks.yml not found in {deployment_dir}")
    dbutils.notebook.exit({"status": "failed", "error": "Bundle config not found"})
else:
    print(f"✅ Found databricks.yml")

In [0]:
print(f"\n{'='*70}")
print("STEP 1: Validate Databricks Bundle")
print(f"{'='*70}\n")

try:
    run_shell_command(
        f"databricks bundle validate -t {target}",
        cwd=deployment_dir
    )
    print("\n✅ Bundle validation passed")
except Exception as e:
    print(f"\n❌ Bundle validation failed: {str(e)}")
    dbutils.notebook.exit({"status": "failed", "error": f"Validation failed: {str(e)}"})

In [0]:
print(f"\n{'='*70}")
print("STEP 2: Deploy Bundle (Idempotent)")
print(f"{'='*70}\n")

print("ℹ️  Note: 'databricks bundle deploy' is idempotent")
print("   - Creates endpoint if it doesn't exist")
print("   - Updates endpoint if config changed")
print("   - No-op if config matches existing state\n")

try:
    run_shell_command(
        f"databricks bundle deploy -t {target}",
        cwd=deployment_dir
    )
    print("\n✅ Bundle deployed successfully")
except Exception as e:
    print(f"\n❌ Bundle deployment failed: {str(e)}")
    dbutils.notebook.exit({"status": "failed", "error": f"Deployment failed: {str(e)}"})

In [0]:
print(f"\n{'='*70}")
print("STEP 3: Check Endpoint Status")
print(f"{'='*70}\n")

# Endpoint name based on target
endpoint_name = f"workday_sales_rag_{target}_endpoint"

try:
    run_shell_command(
        f"databricks serving-endpoints get --name {endpoint_name}",
        cwd=deployment_dir
    )
except subprocess.CalledProcessError:
    print(f"⚠️  Could not retrieve endpoint status (may still be updating)")
    # Don't fail the job if status check fails

In [0]:
print(f"\n{'='*70}")
print(f"✅ ENDPOINT DEPLOYMENT COMPLETE!")
print(f"{'='*70}")
print(f"⏰ Finished at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

print(f"📊 Summary:")
print(f"   Environment: {target.upper()}")
print(f"   Endpoint: {endpoint_name}")
print(f"   Deployment: Idempotent (safe to re-run)")
print(f"\n🔗 View endpoint in Databricks UI:")
print(f"   Serving > Endpoints > {endpoint_name}\n")

# Exit with success status
dbutils.notebook.exit({
    "status": "success",
    "endpoint_name": endpoint_name,
    "target": target,
    "deployment_type": "idempotent_bundle_deploy"
})